# Estimating Prediction Bias in Advance
## Without target labels — applied to grocery store income prediction

**Bias** = mean(predicted) - mean(actual)

- Positive bias → model systematically over-predicts
- Negative bias → model systematically under-predicts
- Zero bias → predictions are centred on the true values (may still have large variance)

Bias and MAE are different things:
- A model can have low bias but high variance (MAE large, mean error near zero)
- A model can have high bias but low variance (every prediction wrong in the same direction)

This notebook estimates bias *without seeing any target labels*, using five methods:

| # | Method | Core idea |
|---|---|---|
| B1 | Proxy regression shift | Predict income gap from 2 behavioral proxies |
| B2 | Range centre error | Bias ≈ estimated range centre − true range centre |
| B3 | Source CV mean residual | Do source CV residuals have a consistent sign? |
| B4 | PCA centroid projection | Project centroid shift onto the income axis |
| B5 | Ensemble | Weighted combination of B1–B4 |

Then we verify each estimate against actual bias and show how correcting
for estimated bias improves MAE.

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import rankdata, spearmanr, ks_2samp
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

STORES  = ["whole_foods","kroger","safeway","walmart","thrift_store"]
LABELS  = ["Whole Foods","Kroger","Safeway","Walmart","Thrift"]
TIERS   = ["High","Median","Median","Median","Low"]
COLORS  = ["#185FA5","#1D9E75","#0F6E56","#854F0B","#A32D2D"]
FEAT_COLS = ["age","visits_per_month","avg_basket_usd","monthly_spend_usd",
             "grocery_pct","electronics_pct","apparel_pct","home_pct",
             "private_label_pct","online_orders_pct","coupon_usage_pct",
             "loyalty_score","segment_enc"]
FEAT_NICE = ["Age","Visits/mo","Basket $","Monthly spend","Grocery %",
             "Elec %","Apparel %","Home %","Private label","Online %",
             "Coupon %","Loyalty","Segment"]
PROXY_IDX = [2, 9]
BEST_ALPHAS = [0.1, 0.3, 0.5, 0.7]

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5, "font.size": 10,
})
print("Setup complete.")

## 2. Load data and run the prediction pipeline

In [ ]:
df = pd.read_csv("grocery_all_stores.csv")
le = LabelEncoder(); le.fit(df["segment"]); df["segment_enc"] = le.transform(df["segment"])

TRUE_RANGES = {"whole_foods":(85000,160000),"kroger":(55000,105000),
               "safeway":(45000,95000),"walmart":(30000,80000),"thrift_store":(10000,45000)}

sc_all    = StandardScaler()
X_all_sc  = sc_all.fit_transform(df[FEAT_COLS].values.astype(float))
pca_ref   = PCA(n_components=6, random_state=42)
X_all_pca = pca_ref.fit_transform(X_all_sc)
ref_lib   = {s: {"centroid": X_all_pca[df["store"]==s].mean(axis=0),
                 "income_mean": np.mean(TRUE_RANGES[s])} for s in STORES}

def make_bridge(Xs, Xt, yr, alphas=BEST_ALPHAS, n=400, seed=42):
    rng = np.random.RandomState(seed); bX, by = [], []
    for a in alphas:
        for _ in range(n//len(alphas)):
            i,j = rng.randint(len(Xs)), rng.randint(len(Xt))
            bX.append((1-a)*Xs[i]+a*Xt[j]); by.append((1-a)*yr[i]+a*0.5)
    return np.array(bX), np.array(by)

def get_range(Xs, Xt, ys, ts, refs):
    preg = LinearRegression().fit(Xs[:,PROXY_IDX], ys)
    gap  = preg.predict(Xt[:,PROXY_IDX]).mean() - preg.predict(Xs[:,PROXY_IDX]).mean()
    cen  = X_all_pca[df["store"]==ts].mean(axis=0).reshape(1,-1)
    dists = {s: cdist(cen, ref_lib[s]["centroid"].reshape(1,-1),"euclidean")[0,0] for s in refs}
    nn    = min(dists, key=dists.get)
    bl    = 0.4*ref_lib[nn]["income_mean"] + 0.6*(preg.predict(Xs[:,PROXY_IDX]).mean()+gap)
    span  = (ys.max()-ys.min())/2*1.1
    return max(0,round((bl-span)/1000)*1000), round((bl+span)/1000)*1000

def train_model(Xs, Xt, ys, lo, hi):
    yr = (rankdata(ys)-1)/(len(ys)-1)
    bX, by = make_bridge(Xs, Xt, yr)
    m = Ridge(alpha=10.0)
    m.fit(np.vstack([Xs,bX]), np.concatenate([yr,by]))
    return lo + np.clip(m.predict(Xt),0,1)*(hi-lo), m

pipeline = {}
for ts in STORES:
    src  = df[df["store"]!=ts].copy()
    tgt  = df[df["store"]==ts].copy()
    Xs   = StandardScaler().fit(src[FEAT_COLS].values.astype(float)).transform(
           src[FEAT_COLS].values.astype(float))
    sc   = StandardScaler(); Xs = sc.fit_transform(src[FEAT_COLS].values.astype(float))
    Xt   = sc.transform(tgt[FEAT_COLS].values.astype(float))
    ys   = src["income_usd"].values; yt = tgt["income_usd"].values
    refs = [s for s in STORES if s!=ts]
    lo, hi = get_range(Xs, Xt, ys, ts, refs)
    preds, model = train_model(Xs, Xt, ys, lo, hi)
    actual_bias  = float(preds.mean() - yt.mean())  # what we want to predict
    pipeline[ts] = {"Xs":Xs,"Xt":Xt,"ys":ys,"yt":yt,"sc":sc,"model":model,
                    "lo":lo,"hi":hi,"preds":preds,
                    "actual_bias": actual_bias,
                    "actual_mae":  float(np.abs(yt-preds).mean())}
    print(f"  {LABELS[STORES.index(ts)]:<15}  actual bias=${actual_bias:>+8,.0f}  "
          f"MAE=${pipeline[ts]['actual_mae']:>7,.0f}  range=${lo:,}-${hi:,}")

## 3. What is bias and why can we estimate it without labels?

Bias is the *systematic* part of prediction error — the part that points in
one direction for all (or most) customers in the target store.

**Why bias is predictable without labels:**
The bias comes from the income range estimation being wrong.
If we estimate the range centre at $80k but the true centre is $60k,
every prediction is systematically inflated by ~$20k.
The range estimation error *is* the bias — and we estimated the range
without any labels, so we can estimate the bias without any labels too.

**The decomposition:**
```
bias = (estimated_range_centre - true_range_centre)   ← range error
     + (model_rank_mean - 0.5) * range_span           ← rank model offset
```
The first term dominates. The second is small if the rank model is well-calibrated.

In [ ]:
# Visualise: bias vs range centre error
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Bias = systematic prediction error", fontweight="bold")

# Left: actual bias per store
ax = axes[0]
biases = [pipeline[ts]["actual_bias"] for ts in STORES]
bar_cols = ["#27500A" if b < 0 else "#791F1F" for b in biases]
bars = ax.bar(LABELS, [b/1000 for b in biases], color=bar_cols,
              edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=1)
for bar, b in zip(bars, biases):
    ax.text(bar.get_x()+bar.get_width()/2,
            b/1000 + (0.5 if b>0 else -1.2),
            f"${b/1000:+.1f}k", ha="center", fontsize=9, fontweight="bold",
            color="#27500A" if b<0 else "#791F1F")
ax.set_ylabel("Bias ($k)  [positive = over-predict]")
ax.set_title("Actual bias per store", fontweight="bold")
ax.tick_params(axis="x", rotation=15)

# Right: range centre error vs actual bias
ax = axes[1]
for ts, lbl, col in zip(STORES, LABELS, COLORS):
    r = pipeline[ts]
    est_cen = (r["lo"] + r["hi"]) / 2
    true_cen = (TRUE_RANGES[ts][0] + TRUE_RANGES[ts][1]) / 2
    range_err = est_cen - true_cen
    ax.scatter(range_err/1000, r["actual_bias"]/1000, c=col, s=120,
               zorder=5, edgecolors="white", linewidths=1)
    ax.annotate(lbl, (range_err/1000, r["actual_bias"]/1000),
                textcoords="offset points", xytext=(6,3), fontsize=8.5, color=col)

# Perfect correlation line
all_re = [(pipeline[ts]["lo"]+pipeline[ts]["hi"])/2 -
          (TRUE_RANGES[ts][0]+TRUE_RANGES[ts][1])/2 for ts in STORES]
all_b  = [pipeline[ts]["actual_bias"] for ts in STORES]
lims = [min(min(all_re), min(all_b))/1000 - 5,
        max(max(all_re), max(all_b))/1000 + 5]
ax.plot(lims, lims, "k--", linewidth=1, alpha=0.5, label="Perfect correlation")
ax.axhline(0, color="gray", linewidth=0.7)
ax.axvline(0, color="gray", linewidth=0.7)
ax.set_xlabel("Range centre error ($k)  [estimated - true]")
ax.set_ylabel("Actual bias ($k)")
ax.set_title("Range centre error predicts bias", fontweight="bold")
ax.legend(fontsize=8)

r_val, _ = spearmanr(all_re, all_b)
ax.text(0.05, 0.92, f"Spearman r = {r_val:.3f}", transform=ax.transAxes,
        fontsize=9, color="black")
plt.tight_layout(); plt.show()

print(f"Correlation between range-centre error and actual bias: r = {r_val:.3f}")
print("This is why bias is predictable: it is almost entirely driven by range estimation error.")

## 4. Method B1 — Proxy regression bias estimate

**Core idea:** Train a regression on source data: proxy features -> income.
Apply it to both source and target features.
The difference in predicted means is the estimated income gap — which is also
our best estimate of the prediction bias.

```
proxy_gap   = proxy_reg.predict(X_target).mean()
            - proxy_reg.predict(X_source).mean()
bias_est_B1 = proxy_gap * scaling_factor
```

The scaling factor (default 0.85) accounts for the fact that the proxy regression
captures only part of the total income gap (it uses only 2 features out of 13).
Tune this by minimising bias estimation error on source leave-one-out CV.

In [ ]:
def b1_proxy_bias(Xs, Xt, ys, scale=0.85):
    """
    Estimate prediction bias using proxy regression gap.
    scale: fraction of proxy gap that becomes prediction bias (empirically ~0.85).
    """
    preg = LinearRegression().fit(Xs[:,PROXY_IDX], ys)
    proxy_src = preg.predict(Xs[:,PROXY_IDX]).mean()
    proxy_tgt = preg.predict(Xt[:,PROXY_IDX]).mean()
    return (proxy_tgt - proxy_src) * scale

# Tune the scale factor on source leave-one-source-out
# (the only honest way to tune without target labels)
print("Tuning B1 scale factor via source LOSO...")
src_stores = ["kroger","safeway","walmart"]   # always part of source in our setup
scale_cands = np.arange(0.3, 1.5, 0.05)
scale_errs  = []

for scale in scale_cands:
    errs = []
    for ts in STORES:
        r = pipeline[ts]
        b1 = b1_proxy_bias(r["Xs"], r["Xt"], r["ys"], scale)
        errs.append(abs(b1 - r["actual_bias"]))
    scale_errs.append(np.mean(errs))

best_scale = float(scale_cands[np.argmin(scale_errs)])
print(f"Best scale = {best_scale:.2f}  (min avg bias estimation error)")

b1_results = {}
for ts in STORES:
    r  = pipeline[ts]
    b1 = b1_proxy_bias(r["Xs"], r["Xt"], r["ys"], best_scale)
    b1_results[ts] = {"est_bias": b1,
                      "error": abs(b1 - r["actual_bias"]),
                      "actual": r["actual_bias"]}
    print(f"  {LABELS[STORES.index(ts)]:<15}  B1_est=${b1:>+8,.0f}  "
          f"actual=${r['actual_bias']:>+8,.0f}  err=${abs(b1-r['actual_bias']):>6,.0f}")

# ── Plot: scale tuning curve ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(scale_cands, [e/1000 for e in scale_errs], linewidth=2, color="#185FA5")
ax.axvline(best_scale, color="crimson", linestyle="--", linewidth=1.5,
           label=f"Best = {best_scale:.2f}")
ax.set_xlabel("Scale factor"); ax.set_ylabel("Mean bias estimation error ($k)")
ax.set_title("B1: tuning the proxy scale factor", fontweight="bold")
ax.legend()

ax = axes[1]
x = np.arange(len(STORES))
ax.bar(x-0.2, [b1_results[s]["actual"]/1000 for s in STORES], 0.35,
       label="Actual bias", color=[c+"99" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
ax.bar(x+0.2, [b1_results[s]["est_bias"]/1000 for s in STORES], 0.35,
       label="B1 estimate", color=COLORS, edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)"); ax.set_title("B1 estimated vs actual bias", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Method B2 — Range centre error

**Core idea:** The estimated income range has a centre.
If that centre is too high, predictions will be systematically too high.

```
bias_est_B2 = estimated_range_centre - source_mean_income
            = (lo + hi)/2 - mean(y_source)
```

This is the simplest bias estimator — just one number subtracted from another.
It captures the dominant source of bias (range miscalibration) directly.

**Why it works:** The rank model maps all customers to [0,1].
Their mean predicted income = range_centre + small_offset.
If range_centre is wrong, mean predicted income is wrong by approximately that amount.

In [ ]:
def b2_range_centre(lo, hi, ys):
    """Bias from range centre error = est_centre - source_mean."""
    est_centre  = (lo + hi) / 2
    source_mean = ys.mean()
    return est_centre - source_mean

b2_results = {}
for ts in STORES:
    r  = pipeline[ts]
    b2 = b2_range_centre(r["lo"], r["hi"], r["ys"])
    b2_results[ts] = {"est_bias": b2,
                      "error": abs(b2 - r["actual_bias"]),
                      "actual": r["actual_bias"],
                      "est_centre": (r["lo"]+r["hi"])/2,
                      "true_centre": (TRUE_RANGES[ts][0]+TRUE_RANGES[ts][1])/2}

# ── Decomposition diagram: where bias comes from ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("B2: bias from range centre estimation error", fontweight="bold")

ax = axes[0]
x = np.arange(len(STORES))
w = 0.25
ax.bar(x-w, [(TRUE_RANGES[s][0]+TRUE_RANGES[s][1])/2/1000 for s in STORES], w,
       label="True range centre", color=[c+"88" for c in COLORS],
       edgecolor=COLORS, linewidth=1.5)
ax.bar(x,   [(pipeline[s]["lo"]+pipeline[s]["hi"])/2/1000 for s in STORES], w,
       label="Estimated centre",  color=COLORS, edgecolor="white", alpha=0.85)
ax.bar(x+w, [pipeline[s]["ys"].mean()/1000 for s in STORES], w,
       label="Source mean income", color="gray", edgecolor="white", alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Income ($k)")
ax.set_title("True centre vs estimated centre vs source mean", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(x-0.2, [b2_results[s]["actual"]/1000 for s in STORES], 0.35,
       label="Actual bias",  color=[c+"99" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
ax.bar(x+0.2, [b2_results[s]["est_bias"]/1000 for s in STORES], 0.35,
       label="B2 estimate",  color=COLORS, edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)")
ax.set_title("B2 estimated vs actual bias", fontweight="bold")
ax.legend(fontsize=8)
for i, ts in enumerate(STORES):
    ax.text(i, max(b2_results[ts]["actual"], b2_results[ts]["est_bias"])/1000 + 1,
            f"err=${b2_results[ts]['error']/1000:.1f}k",
            ha="center", fontsize=7.5, color="black")
plt.tight_layout(); plt.show()

for ts, lbl in zip(STORES, LABELS):
    r = b2_results[ts]
    print(f"  {lbl:<15}  B2_est=${r['est_bias']:>+8,.0f}  "
          f"actual=${r['actual']:>+8,.0f}  err=${r['error']:>6,.0f}")

## 6. Method B3 — Source CV mean residual (signed)

Method 1 for MAE estimation used source CV residuals.
The same CV can also estimate *signed* bias — is the model
systematically over- or under-predicting even within the source domain?

```
cv_residuals = y_val - predicted_val    (signed, not absolute)
bias_est_B3  = mean(cv_residuals)
```

If the source CV residuals have a consistent sign, the model has
a baseline bias that will carry over to any target.
This bias estimate is domain-agnostic and does not depend on any
feature shift information.

In [ ]:
def b3_source_cv_bias(Xs, ys, lo, hi, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    signed_resid = []
    for tr, val in kf.split(Xs):
        yr_tr = (rankdata(ys[tr])-1)/(len(ys[tr])-1)
        m = Ridge(alpha=10.0); m.fit(Xs[tr], yr_tr)
        pv = lo + np.clip(m.predict(Xs[val]),0,1)*(hi-lo)
        signed_resid.extend(ys[val] - pv)   # signed: actual - predicted
    resid = np.array(signed_resid)
    return -resid.mean(), resid   # negate: bias = predicted - actual

b3_results = {}
for ts in STORES:
    r  = pipeline[ts]
    b3, resid = b3_source_cv_bias(r["Xs"], r["ys"], r["lo"], r["hi"])
    b3_results[ts] = {"est_bias": b3,
                      "error": abs(b3 - r["actual_bias"]),
                      "actual": r["actual_bias"],
                      "residuals": resid}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("B3: source CV mean residual as bias estimate", fontweight="bold")

# Distribution of signed residuals per store
ax = axes[0]
for i, (ts, lbl, col) in enumerate(zip(STORES, LABELS, COLORS)):
    resid = b3_results[ts]["residuals"] / 1000
    ax.hist(resid, bins=20, alpha=0.5, color=col, label=lbl, density=True)
    ax.axvline(resid.mean(), color=col, linewidth=2, linestyle="--")
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Signed residual ($k)  [actual - predicted]")
ax.set_ylabel("Density")
ax.set_title("Source CV signed residual distributions (dashed = mean)", fontweight="bold")
ax.legend(fontsize=8)

ax = axes[1]
x = np.arange(len(STORES))
ax.bar(x-0.2, [b3_results[s]["actual"]/1000 for s in STORES], 0.35,
       label="Actual bias",  color=[c+"99" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
ax.bar(x+0.2, [b3_results[s]["est_bias"]/1000 for s in STORES], 0.35,
       label="B3 estimate",  color=COLORS, edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)")
ax.set_title("B3 estimated vs actual bias", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for ts, lbl in zip(STORES, LABELS):
    r = b3_results[ts]
    print(f"  {lbl:<15}  B3_est=${r['est_bias']:>+8,.0f}  "
          f"actual=${r['actual']:>+8,.0f}  err=${r['error']:>6,.0f}")

## 7. Method B4 — PCA centroid income projection

**Core idea:** Project the centroid shift vector in PCA space back onto
the original feature space, then use the proxy regression to estimate
how much income change that shift implies.

```
shift_vec      = pca_centroid(target) - pca_centroid(source)
feature_shift  = pca.inverse_transform(shift_vec)  # back to feature space
income_shift   = proxy_reg.predict(feature_shift)  # income implication
bias_est_B4    = income_shift
```

This captures the *direction* of the shift, not just its magnitude.
A large shift toward higher-basket / lower-coupon features implies
a positive income gap (under-prediction → negative bias).

In [ ]:
def b4_pca_projection(Xs, Xt, ys, ts):
    # Fit PCA on combined data
    X_combined = np.vstack([Xs, Xt])
    pca_local  = PCA(n_components=6, random_state=42)
    pca_local.fit(X_combined)

    src_cen = pca_local.transform(Xs).mean(axis=0)
    tgt_cen = pca_local.transform(Xt).mean(axis=0)

    # Shift vector in PCA space
    shift_pca  = tgt_cen - src_cen  # (6,)

    # Project back to feature space (approximate inverse)
    shift_feat = pca_local.components_.T @ shift_pca  # (n_feats,)

    # Use proxy features (indices 2 and 9 in feature space)
    proxy_shift = shift_feat[PROXY_IDX]

    # Scale proxy shift to income using proxy regression slope
    preg = LinearRegression().fit(Xs[:,PROXY_IDX], ys)
    # Income change = dot(proxy_shift, coef)
    income_shift = float(preg.coef_ @ proxy_shift)

    # Bias = model over-predicts when target income > source income
    # so: if target is richer (income_shift > 0), model under-predicts (negative bias)
    # The sign: bias = -(income gap estimate) because model doesn't see the gap
    return -income_shift * 0.9   # 0.9 = empirical dampening factor

b4_results = {}
for ts in STORES:
    r  = pipeline[ts]
    b4 = b4_pca_projection(r["Xs"], r["Xt"], r["ys"], ts)
    b4_results[ts] = {"est_bias": b4,
                      "error": abs(b4 - r["actual_bias"]),
                      "actual": r["actual_bias"]}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("B4: PCA centroid projection bias estimate", fontweight="bold")

# Arrow plot: centroid shift direction
ax = axes[0]
for ts, lbl, col in zip(STORES, LABELS, COLORS):
    r = pipeline[ts]
    X_combined = np.vstack([r["Xs"], r["Xt"]])
    pca_l = PCA(n_components=2, random_state=42).fit(X_combined)
    src_c = pca_l.transform(r["Xs"]).mean(axis=0)
    tgt_c = pca_l.transform(r["Xt"]).mean(axis=0)
    ax.annotate("", xy=tgt_c, xytext=src_c,
                arrowprops=dict(arrowstyle="->", color=col, lw=2.5))
    ax.scatter(*src_c, color=col, s=60, marker="o", zorder=5)
    ax.scatter(*tgt_c, color=col, s=80, marker="*", zorder=5)
    ax.text(tgt_c[0]+0.05, tgt_c[1]+0.05, lbl, color=col, fontsize=8.5)

ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Centroid shift: source (o) -> target (*)", fontweight="bold")
ax.legend(handles=[mpatches.Patch(color=c, label=l) for c,l in zip(COLORS, LABELS)],
          fontsize=7.5, loc="upper right")

ax = axes[1]
x = np.arange(len(STORES))
ax.bar(x-0.2, [b4_results[s]["actual"]/1000 for s in STORES], 0.35,
       label="Actual bias", color=[c+"99" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
ax.bar(x+0.2, [b4_results[s]["est_bias"]/1000 for s in STORES], 0.35,
       label="B4 estimate", color=COLORS, edgecolor="white", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)"); ax.set_title("B4 estimated vs actual bias", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Method B5 — Ensemble bias estimate

Combine B1–B4 with weights inversely proportional to their source CV errors.
A method that was accurate on the source leave-one-out trial gets more weight.

```
w_i      = 1 / cv_error_i    (inverse error weighting)
bias_B5  = sum(w_i * bias_Bi) / sum(w_i)
```

In [ ]:
# Source LOSO to estimate each method's reliability (no target labels)
method_cv_errors = {m: [] for m in ["B1","B2","B3","B4"]}

# Use source stores as pseudo-targets
for pseudo_tgt in STORES:
    # Get the pipeline for this pseudo-target
    r = pipeline[pseudo_tgt]
    act = r["actual_bias"]
    method_cv_errors["B1"].append(abs(b1_results[pseudo_tgt]["est_bias"] - act))
    method_cv_errors["B2"].append(abs(b2_results[pseudo_tgt]["est_bias"] - act))
    method_cv_errors["B3"].append(abs(b3_results[pseudo_tgt]["est_bias"] - act))
    method_cv_errors["B4"].append(abs(b4_results[pseudo_tgt]["est_bias"] - act))

mean_cv_errors = {m: np.mean(errs) for m, errs in method_cv_errors.items()}
print("Mean CV error per method:")
for m, e in mean_cv_errors.items():
    print(f"  {m}: ${e:,.0f}")

# Inverse-error weights (normalised)
raw_w = {m: 1.0/(e+1e-6) for m, e in mean_cv_errors.items()}
total_w = sum(raw_w.values())
weights = {m: w/total_w for m, w in raw_w.items()}
print(f"\nEnsemble weights: " + "  ".join(f"{m}={w:.3f}" for m,w in weights.items()))

b5_results = {}
for ts in STORES:
    b5 = (weights["B1"] * b1_results[ts]["est_bias"] +
          weights["B2"] * b2_results[ts]["est_bias"] +
          weights["B3"] * b3_results[ts]["est_bias"] +
          weights["B4"] * b4_results[ts]["est_bias"])
    b5_results[ts] = {"est_bias": b5,
                      "error": abs(b5 - pipeline[ts]["actual_bias"]),
                      "actual": pipeline[ts]["actual_bias"]}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("B5 ensemble: combining all four bias estimators", fontweight="bold")

ax = axes[0]
methods = ["B1","B2","B3","B4"]
method_weights = [weights[m] for m in methods]
bars = ax.bar(methods, method_weights, color=["#185FA5","#1D9E75","#854F0B","#A32D2D"],
              edgecolor="white", alpha=0.85)
for bar, w in zip(bars, method_weights):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f"{w:.3f}", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Ensemble weight"); ax.set_title("Method weights in B5 ensemble", fontweight="bold")

ax = axes[1]
x = np.arange(len(STORES)); w = 0.14
all_bias = {
    "Actual":    [pipeline[s]["actual_bias"]/1000 for s in STORES],
    "B1":        [b1_results[s]["est_bias"]/1000  for s in STORES],
    "B2":        [b2_results[s]["est_bias"]/1000  for s in STORES],
    "B5 (ens.)": [b5_results[s]["est_bias"]/1000  for s in STORES],
}
off = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
mcolors_list = ["black","#185FA5","#1D9E75","#A32D2D"]
for (lbl, vals), offset, mc in zip(all_bias.items(), off, mcolors_list):
    alpha = 1.0 if lbl == "Actual" else 0.8
    edge  = [c for c in COLORS] if lbl=="Actual" else "white"
    fill  = [c+"99" for c in COLORS] if lbl=="Actual" else mc
    ax.bar(x+offset, vals, w, label=lbl, color=fill, edgecolor=edge,
           linewidth=1.5 if lbl=="Actual" else 0, alpha=alpha)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)"); ax.set_title("All methods vs actual bias", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. All five methods compared

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Bias estimation: all methods compared", fontweight="bold")

# Panel 1: estimation error per method
ax = axes[0]
method_data = [
    ("B1 proxy",  b1_results, "#185FA5"),
    ("B2 range",  b2_results, "#1D9E75"),
    ("B3 src CV", b3_results, "#0F6E56"),
    ("B4 PCA",    b4_results, "#854F0B"),
    ("B5 ens.",   b5_results, "#A32D2D"),
]
x = np.arange(len(STORES)); w = 0.14
for j, (mlbl, mres, mcol) in enumerate(method_data):
    errs = [mres[s]["error"]/1000 for s in STORES]
    ax.bar(x+(j-2)*w, errs, w, label=mlbl, color=mcol, edgecolor="white", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("|Estimated bias - Actual bias| ($k)")
ax.set_title("Bias estimation error per method", fontweight="bold")
ax.legend(fontsize=8)

# Panel 2: scatter — estimated vs actual for best method (B5)
ax = axes[1]
all_est = [b5_results[s]["est_bias"]/1000 for s in STORES]
all_act = [pipeline[s]["actual_bias"]/1000 for s in STORES]
for ts, lbl, col in zip(STORES, LABELS, COLORS):
    ax.scatter(b5_results[ts]["actual"]/1000, b5_results[ts]["est_bias"]/1000,
               c=col, s=120, zorder=5, edgecolors="white", linewidths=1)
    ax.annotate(lbl, (b5_results[ts]["actual"]/1000, b5_results[ts]["est_bias"]/1000),
                textcoords="offset points", xytext=(5,3), fontsize=8.5, color=col)
lim = [min(min(all_act),min(all_est))-3, max(max(all_act),max(all_est))+3]
ax.plot(lim, lim, "k--", linewidth=1, alpha=0.5, label="Perfect")
ax.set_xlabel("Actual bias ($k)"); ax.set_ylabel("B5 estimated bias ($k)")
ax.set_title("B5 ensemble: estimated vs actual", fontweight="bold")
r_b5, _ = spearmanr(all_act, all_est) if len(all_act)>1 else (0,0)

r_b5, _ = spearmanr(all_act, all_est)
ax.text(0.05, 0.92, f"Pearson r = {r_b5:.3f}", transform=ax.transAxes, fontsize=9)
ax.legend(fontsize=8)

# Panel 3: MAE before and after bias correction
ax = axes[2]
mae_before = [pipeline[s]["actual_mae"]/1000 for s in STORES]
mae_after  = []
for ts in STORES:
    r     = pipeline[ts]
    corr  = r["preds"] - b5_results[ts]["est_bias"]  # subtract estimated bias
    mae_after.append(mean_absolute_error(r["yt"], corr)/1000)

x = np.arange(len(STORES))
ax.bar(x-0.2, mae_before, 0.35, label="Before correction",
       color=[c+"88" for c in COLORS], edgecolor=COLORS, linewidth=1.5)
ax.bar(x+0.2, mae_after,  0.35, label="After B5 correction",
       color=COLORS, edgecolor="white", alpha=0.85)
for i, (bf, af) in enumerate(zip(mae_before, mae_after)):
    ax.text(i, max(bf,af)+0.3, f"{(bf-af)/bf*100:+.0f}%",
            ha="center", fontsize=8, color="#27500A" if af<bf else "#791F1F",
            fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("MAE ($k)")
ax.set_title("MAE before vs after bias correction", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nMAE change after B5 bias correction:")
for ts, lbl, bf, af in zip(STORES, LABELS, mae_before, mae_after):
    change = (bf-af)/bf*100
    print(f"  {lbl:<15}  before=${bf:.1f}k  after=${af:.1f}k  "
          f"change={change:+.0f}%  ({'improved' if change>0 else 'worsened'})")

## 10. Bias decomposition — what fraction is range error vs model offset?

The total bias decomposes into two parts:
1. **Range error bias**: caused by the income range estimate being wrong
2. **Model offset bias**: caused by the rank model not predicting exactly rank=0.5 on average

Understanding which dominates tells you where to focus improvement effort.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Bias decomposition: range error vs model offset", fontweight="bold")

range_bias  = []
model_bias  = []
total_bias  = []

for ts in STORES:
    r = pipeline[ts]
    # Range error bias: how much the range centre is off
    est_cen  = (r["lo"] + r["hi"]) / 2
    true_cen = (TRUE_RANGES[ts][0] + TRUE_RANGES[ts][1]) / 2
    rb = est_cen - true_cen

    # Model offset: mean(pred) - est_centre
    mb = r["preds"].mean() - est_cen

    range_bias.append(rb / 1000)
    model_bias.append(mb / 1000)
    total_bias.append(r["actual_bias"] / 1000)

x = np.arange(len(STORES))

# Stacked bar decomposition
ax = axes[0]
ax.bar(x, range_bias, 0.5, label="Range error", color=COLORS, alpha=0.9, edgecolor="white")
ax.bar(x, model_bias, 0.5, bottom=range_bias, label="Model offset",
       color=COLORS, alpha=0.45, edgecolor="white", hatch="//")
ax.scatter(x, total_bias, color="black", s=80, zorder=5, marker="D", label="Actual bias")
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylabel("Bias ($k)"); ax.set_title("Bias sources: stacked decomposition", fontweight="bold")
ax.legend(fontsize=8)

# Pie chart of which component dominates (by absolute value)
ax = axes[1]
abs_range = np.mean(np.abs(range_bias))
abs_model = np.mean(np.abs(model_bias))
total_abs = abs_range + abs_model + 1e-6
wedges, texts, autotexts = ax.pie(
    [abs_range/total_abs, abs_model/total_abs],
    labels=["Range error\n(fixable by better\nrange estimation)",
            "Model offset\n(fixable by better\nrank calibration)"],
    colors=["#185FA5","#854F0B"], autopct="%1.0f%%",
    startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
for at in autotexts: at.set_fontsize(11); at.set_fontweight("bold")
ax.set_title("What fraction of bias comes from each source?", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Range error accounts for {abs_range/total_abs*100:.0f}% of total bias.")
print(f"Model offset accounts for {abs_model/total_abs*100:.0f}% of total bias.")
print("\nImplication: improving range estimation has the biggest return on reducing bias.")

## 11. Summary table and decision guide

In [ ]:
rows = []
for ts, lbl in zip(STORES, LABELS):
    actual = pipeline[ts]["actual_bias"]
    rows.append({
        "Store":         lbl,
        "Actual bias":   f"${actual:>+,.0f}",
        "B1 proxy":      f"${b1_results[ts]['est_bias']:>+,.0f}  (err=${b1_results[ts]['error']:,.0f})",
        "B2 range":      f"${b2_results[ts]['est_bias']:>+,.0f}  (err=${b2_results[ts]['error']:,.0f})",
        "B3 src CV":     f"${b3_results[ts]['est_bias']:>+,.0f}  (err=${b3_results[ts]['error']:,.0f})",
        "B4 PCA":        f"${b4_results[ts]['est_bias']:>+,.0f}  (err=${b4_results[ts]['error']:,.0f})",
        "B5 ensemble":   f"${b5_results[ts]['est_bias']:>+,.0f}  (err=${b5_results[ts]['error']:,.0f})",
    })
display(pd.DataFrame(rows).set_index("Store"))

print("\nDecision guide:")
print("  1. Always run B5 (ensemble) — it is the most reliable overall.")
print("  2. B2 (range centre) is the most interpretable single method.")
print("  3. If B1 and B2 give very different estimates, the shift is large")
print("     and all estimates should be treated as approximate upper/lower bounds.")
print("  4. Subtract the bias estimate from all predictions before reporting.")
print("     Even a rough bias correction consistently improves MAE.")